# Mini-tutorial: Looking inside a fitted Pipeline

After:

```python
model.fit(X, y)
```

we can inspect what the pipeline actually learned:

- imputer medians
- scaler means and scales
- one-hot categories
- the transformed design matrix
- regression intercept
- regression coefficients


## 0. Setup

Put `mystery_dataset_1.csv` in the same folder as this notebook.


In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

df = pd.read_csv("mystery_dataset_1.csv")


## 1. Build a small example

We'll predict `Step_Height` from a few numeric and categorical features.


In [2]:
numeric_features = [
    "Belt_Speed",
    "Step_Length",
    "Mean_Ankle",
]

categorical_features = [
    "Speed_Group",
    "Timepoint_Name",
]

features = numeric_features + categorical_features
target = "Step_Height"


In [3]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("regression", LinearRegression()),
])


## 2. Fit it


In [4]:
tiny = df.dropna(subset=[target]).head(500).copy()

X_tiny = tiny[features]
y_tiny = tiny[target]

model.fit(X_tiny, y_tiny)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['Belt_Speed','Step_Length','Mean_Ankle','Speed_Group','Timepoint_Name']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This sub

## 3. `get_params()` is configuration, not learned values

`model.get_params()` shows settings and hyperparameters.

Learned quantities usually appear in attributes ending with `_`, such as:

```text
coef_
intercept_
mean_
scale_
statistics_
categories_
```


In [5]:
model.get_params()


{'memory': None,
 'steps': [('preprocess',
   ColumnTransformer(transformers=[('numeric',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('scaler', StandardScaler())]),
                                    ['Belt_Speed', 'Step_Length', 'Mean_Ankle']),
                                   ('categorical',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehot',
                                                     OneHotEncoder(handle_unknown='ignore',
                                                                   sparse_output=False))]),
                                    ['Speed_Group', 'Timepoint_Name'])])),
  ('regression', LinearRegression())],
 'transform_input': N

## 4. Inspect the fitted regression


In [6]:
reg = model.named_steps["regression"]

print("Intercept:")
print(reg.intercept_)

print("\nRaw coefficient array:")
print(reg.coef_)


Intercept:
19.650100595353248

Raw coefficient array:
[ 24.26508662  20.05105507   2.15393236  34.0390119   18.17521188
   0.65581039 -15.26742824 -37.60260593   4.20166717  -4.20166717]


The raw coefficient array is hard to interpret until we recover the transformed feature names.


In [7]:
pre = model.named_steps["preprocess"]

feature_names = pre.get_feature_names_out()

feature_names


array(['numeric__Belt_Speed', 'numeric__Step_Length',
       'numeric__Mean_Ankle', 'categorical__Speed_Group_speed16',
       'categorical__Speed_Group_speed20',
       'categorical__Speed_Group_speed24',
       'categorical__Speed_Group_speed28',
       'categorical__Speed_Group_speed32',
       'categorical__Timepoint_Name_week-1',
       'categorical__Timepoint_Name_week4'], dtype=object)

In [8]:
coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": reg.coef_,
})

coef_table["abs_coefficient"] = coef_table["coefficient"].abs()

coef_table.sort_values("abs_coefficient", ascending=False)


,feature,coefficient,abs_coefficient
7,categorical__Speed_Group_speed32,-37.602606,37.602606
3,categorical__Speed_Group_speed16,34.039012,34.039012
0,numeric__Belt_Speed,24.265087,24.265087
1,numeric__Step_Length,20.051055,20.051055
4,categorical__Speed_Group_speed20,18.175212,18.175212
6,categorical__Speed_Group_speed28,-15.267428,15.267428
9,categorical__Timepoint_Name_week4,-4.201667,4.201667
8,categorical__Timepoint_Name_week-1,4.201667,4.201667
2,numeric__Mean_Ankle,2.153932,2.153932
5,categorical__Speed_Group_speed24,0.655810,0.655810


The largest coefficient is **not automatically the most biologically important variable**. Scaling, coding, redundancy, and collinearity all affect coefficient interpretation.


## 5. See the actual matrix sent to LinearRegression


In [9]:
X_transformed = pre.transform(X_tiny)

print("Original shape:", X_tiny.shape)
print("Transformed shape:", X_transformed.shape)


Original shape: (500, 5)
Transformed shape: (500, 10)


In [10]:
X_transformed_df = pd.DataFrame(
    X_transformed,
    columns=feature_names,
    index=X_tiny.index,
)

X_transformed_df.head()


,numeric__Belt_Speed,numeric__Step_Length,numeric__Mean_Ankle,categorical__Speed_Group_speed16,categorical__Speed_Group_speed20,categorical__Speed_Group_speed24,categorical__Speed_Group_speed28,categorical__Speed_Group_speed32,categorical__Timepoint_Name_week-1,categorical__Timepoint_Name_week4
0,-1.542091,-0.213050,-0.786253,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-1.542091,-0.249266,-1.539511,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-1.542091,-0.121522,-0.922159,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-1.542091,-0.355276,-0.697669,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-1.542091,-0.264590,-1.422948,1.0,0.0,0.0,0.0,0.0,1.0,0.0


Conceptually:

```text
original DataFrame
       ↓
ColumnTransformer
       ↓
numeric design matrix X
       ↓
LinearRegression
       ↓
intercept + coefficients
```


## 6. Inspect what the numerical branch learned


In [11]:
num_pipe = (
    model
    .named_steps["preprocess"]
    .named_transformers_["numeric"]
)

num_pipe


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['Belt_Speed','Step_Length','Mean_Ankle']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=Tr

### Medians learned by the imputer


In [12]:
pd.Series(
    num_pipe.named_steps["imputer"].statistics_,
    index=numeric_features,
    name="learned_median",
)


Belt_Speed      0.240000
Step_Length    10.927813
Mean_Ankle     41.035709
Name: learned_median, dtype: float64

### Means learned by the scaler


In [13]:
pd.Series(
    num_pipe.named_steps["scaler"].mean_,
    index=numeric_features,
    name="learned_mean",
)


Belt_Speed      0.243960
Step_Length    30.055691
Mean_Ankle     46.270493
Name: learned_mean, dtype: float64

### Scale values learned by the scaler


In [14]:
pd.Series(
    num_pipe.named_steps["scaler"].scale_,
    index=numeric_features,
    name="learned_scale",
)


Belt_Speed      0.054446
Step_Length    74.183655
Mean_Ankle     16.592775
Name: learned_scale, dtype: float64

## 7. Inspect what the categorical branch learned


In [15]:
cat_pipe = (
    model
    .named_steps["preprocess"]
    .named_transformers_["categorical"]
)

encoder = cat_pipe.named_steps["onehot"]

for col, categories in zip(
    categorical_features,
    encoder.categories_,
):
    print(f"{col}:")
    print(categories)
    print()


Speed_Group:
['speed16' 'speed20' 'speed24' 'speed28' 'speed32']

Timepoint_Name:
['week-1' 'week4']



Those category lists were learned during `.fit()`. They determine which one-hot indicator columns get created.


## 8. Predictions from the full fitted pipeline


In [16]:
predictions = model.predict(X_tiny.head(5))

pd.DataFrame({
    "observed_Step_Height": y_tiny.head(5).to_numpy(),
    "predicted_Step_Height": predictions,
})


,observed_Step_Height,predicted_Step_Height
0,18.737377,14.506390
1,14.698949,12.157749
2,17.096214,16.048900
3,16.034194,11.845423
4,9.898764,12.101557


## 9. Useful scikit-learn convention

A fitted attribute ending in `_` usually means:

> this value was learned during `.fit()`

Examples:

```text
imputer.statistics_
scaler.mean_
scaler.scale_
encoder.categories_
regression.coef_
regression.intercept_
```


## 10. One-hot encoding and coefficient interpretation

Our encoder currently keeps all categories. With a regression intercept, those one-hot columns are redundant.

Later, if we specifically want cleaner category-coefficient interpretation, we can use:

```python
OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False,
)
```

For now, that redundancy is useful because it leads naturally into the next lesson on **collinearity**.


## 10-minute challenge

Without looking back:

1. Print the fitted median for `Mean_Ankle`.
2. Print the categories learned for `Speed_Group`.
3. Build a table pairing each transformed feature with its regression coefficient.
4. Show the first five rows of the transformed design matrix.
5. Which fitted attributes end in `_`, and why?
